# AutoShift – Intelligent Shift Automation Platform (Story-Telling Version)

### Problem Statement

**"The client had a support team responsible for managing workforce shifts. Whenever a client needed a new shift, update, cancellation, or extension, they would send an email to the support team. The support team manually read the email, identified the client, location, qualification, dates, and timings, then logged into the workforce management system to perform the required action. This process was slow, error-prone, and highly dependent on human interpretation, especially when emails were unstructured or incomplete."**

---

### Business Challenge

**"The same client could refer to locations and qualifications using different names. For example, 'Java Developer', 'Senior Java Engineer', and 'Java Resource' could all refer to the same qualification. Support agents spent significant time searching master data, validating information, and correcting mistakes. The business wanted to automate the entire shift lifecycle while maintaining accuracy and compliance."**

---

### Solution Approach

**"I designed and developed AutoShift, an enterprise-grade Agentic AI platform that automatically converts natural language emails into workforce management actions such as shift creation, updates, cancellations, and extensions."**

---

## Architecture Flow

### Step 1: Email Ingestion

**"When a client sends an email, the email is stored in S3 and an event containing the S3 path is published to RabbitMQ. I used RabbitMQ to decouple email ingestion from processing and support asynchronous scalability."**

```text
Client Email
      ↓
S3 Storage
      ↓
RabbitMQ
      ↓
AutoShift Consumer
```

---

### Step 2: Email Extraction

**"The consumer downloads the email from S3 and extracts the email body, subject, sender information, and thread metadata for further processing."**

```text
Download Email
      ↓
Extract Content
      ↓
Normalize Text
```

---

### Step 3: Master Data Knowledge Base

**"A major challenge was resolving client-specific locations and qualifications. Instead of relying on keyword matching, I built a semantic retrieval layer."**

**"Master data such as clients, locations, and qualifications was fetched from existing .NET APIs."**

```text
.NET APIs
      ↓
Client Master Data
Qualification Master Data
```

---

### Step 4: Custom Chunking & Vectorization

**"Since the source data was JSON-based, traditional document chunking was not effective. I designed custom chunking logic that preserved business entities and relationships."**

Example:

```json
{
  "client":"ABC",
  "location":"Pune",
  "qualification":"Java Developer",
  "id":"101"
}
```

**"These chunks were embedded using Cohere English embeddings and stored in Qdrant along with metadata."**

```text
Master Data
      ↓
Custom Chunking
      ↓
Cohere Embeddings
      ↓
Qdrant Vector DB
```

---

### Step 5: Semantic Retrieval

**"The extracted email was converted into an embedding using the same embedding model. I performed semantic search against Qdrant with metadata filtering, Top-K retrieval, and reranking to identify the most relevant client, location, and qualification information."**

```text
Email
     ↓
Embedding
     ↓
Qdrant Search
     ↓
Top-K Results
     ↓
Reranking
```

---

### Step 6: Intent Detection & Structured Extraction

**"The retrieved business context, system prompt, and original email content were sent to Claude Sonnet through AWS Bedrock."**

**"The model identified the business intent and generated a deterministic JSON response."**

Example:

```json
{
  "intent": "CREATE_SHIFT",
  "client_name": "ABC",
  "location": "Pune",
  "qualification": "Java Developer",
  "date": "2026-08-10",
  "start_time": "09:00",
  "end_time": "18:00"
}
```

**"I used Pydantic output parsers to enforce schema validation and ensure consistent outputs."**

---

### Step 7: Validation Layer

**"Before performing any business action, the extracted data was cross-validated against the master data to prevent hallucinations and invalid requests."**

Validation checks included:

- Client validation
- Location validation
- Qualification validation
- Date validation
- Mandatory field validation

---

### Step 8: Agentic Tool Execution

**"Once validation passed, the agent selected the appropriate tool based on the detected intent."**

Examples:

| Intent | Tool |
|----------|----------|
| CREATE_SHIFT | Create Shift API |
| UPDATE_SHIFT | Update Shift API |
| DELETE_SHIFT | Delete Shift API |
| EXTEND_SHIFT | Extend Shift API |

```text
Intent
   ↓
Tool Selection
   ↓
Downstream API
   ↓
Business Action
```

---

## Human-in-the-Loop (HIL)

**"A key requirement was ensuring business accuracy. If the model detected ambiguity, low confidence, or missing information, the workflow automatically switched to Human-in-the-Loop mode."**

Examples:

```text
Missing Location
Unknown Qualification
Ambiguous Date
Conflicting Information
```

**"The system generated clarification emails to the support team. Once the support team responded, the email thread was reprocessed and continued from the same workflow state."**

```text
Ambiguous Request
        ↓
Clarification Email
        ↓
Support Team Response
        ↓
Workflow Resumes
```

---

## Learning & Continuous Improvement

**"To reduce future manual intervention, I implemented correction learning and alias learning."**

Example:

```text
"Java Resource"
      ↓
Mapped To
      ↓
"Java Developer"
```

**"These mappings were stored and reused in future requests, continuously improving automation accuracy."**

---

## Observability & Monitoring

**"I implemented full observability using LangSmith and Elasticsearch."**

Tracked:

- Prompt execution
- Retrieval quality
- Tool calls
- Agent decisions
- API responses
- Validation failures
- Human intervention events

---

## Reliability & Production Readiness

**"To make the platform enterprise-ready, I implemented:"**

- LangChain tool calling
- Guardrails
- Schema validation
- Model fallback strategy
- Retry mechanisms
- Async FastAPI services
- Event-driven architecture
- API rate limiting
- Security controls

---

## Deployment

**"The platform was containerized using Docker and deployed on AWS ECS. Traffic was routed through API Gateway and Load Balancers to support scalability and high availability."**

```text
API Gateway
      ↓
Load Balancer
      ↓
AWS ECS
      ↓
FastAPI Services
```

---

## Evaluation

**"I evaluated the solution using RAGAS and DeepEval with golden test datasets. We measured retrieval quality, faithfulness, answer relevance, and end-to-end workflow accuracy before production rollout."**

---

## Business Impact

**"The solution automated the complete shift lifecycle process from email ingestion to downstream execution. It reduced manual effort by approximately 65%, improved processing speed, reduced operational errors, and enabled support teams to focus on exception handling rather than repetitive tasks."**

---

## 90-Second Interview Answer

**"At Xeople, I built AutoShift, an Agentic AI platform that automates workforce shift operations from natural language emails. Previously, support teams manually interpreted emails and created or modified shifts in downstream systems. I designed an event-driven architecture where emails were stored in S3, RabbitMQ triggered processing, and FastAPI services handled extraction and orchestration. To resolve client-specific locations and qualifications, I built a RAG layer using custom JSON chunking, Cohere embeddings, and Qdrant vector search. Retrieved context and email content were passed to Claude Sonnet on AWS Bedrock, which generated deterministic structured outputs using Pydantic schemas. After validation against master data, the agent selected tools to create, update, or delete shifts through downstream APIs. For ambiguous requests, I implemented Human-in-the-Loop workflows with clarification emails and correction learning. The solution was deployed on AWS ECS with full observability through LangSmith and Elasticsearch and evaluated using RAGAS and DeepEval. The platform reduced manual shift management effort by approximately 65% while improving speed and accuracy."**